# CS4: Comprehensive Statistical Analysis Framework

This notebook implements three advanced statistical methodologies:
1. **F-tests** for variance equality (homoscedasticity) with significance levels
2. **AR(4) models** with impulse response half-life calculation  
3. **RMSE** calculation using rolling prediction methodology

Data: 6 indicators comparing Iceland to multiple groups (Eurozone, Small Open Economies, Baltics)

Source extraction points:
- F-test: cs4_statistical_analysis.py lines 202-216
- AR(4) model: cs4_statistical_analysis.py lines 277-281
- Half-life: cs4_statistical_analysis.py lines 296-340
- RMSE: cs4_statistical_analysis.py lines 335-384

## 1. Setup and Imports

In [8]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.tsa.ar_model import AutoReg
import warnings
from pathlib import Path
import sys

# Add stats_core library
sys.path.append('../lib')
from stats_core import fit_ar4_model, get_significance_stars

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

print("CS4 Statistical Analysis Framework")
print("="*50)

CS4 Statistical Analysis Framework


## 2. Data Loading

CS4 uses specialized datasets with multiple aggregation methods for comparator groups

In [9]:
# Define data directory
data_dir = Path('../data/Clean/CS4_Statistical_Modeling')

# Define indicators and file mappings
indicators = {
    'Net Direct Investment': 'net_direct_investment_full.csv',
    'Net Portfolio Investment': 'net_portfolio_investment_full.csv',
    'Net Portfolio Investment - Debt Securities': 'net_debt_portfolio_investment_full.csv',
    'Net Portfolio Investment - Equity & Investment Fund Shares': 'net_equity_portfolio_investment_full.csv',
    'Net Other Investment': 'net_other_investment_full.csv',
    'Net Capital Flows': 'net_capital_flows_full.csv'
}

# Define comparator groups
comparator_groups = [
    'eurozone_pgdp_weighted',
    'eurozone_pgdp_simple',
    'soe_pgdp_weighted',
    'soe_pgdp_simple',
    'baltics_pgdp_weighted',
    'baltics_pgdp_simple'
]

# Group labels for reporting
group_labels = {
    'eurozone_pgdp_weighted': 'Eurozone Weighted Avg',
    'eurozone_pgdp_simple': 'Eurozone Simple Avg',
    'soe_pgdp_weighted': 'SOE Weighted Avg',
    'soe_pgdp_simple': 'SOE Simple Avg',
    'baltics_pgdp_weighted': 'Baltics Weighted Avg',
    'baltics_pgdp_simple': 'Baltics Simple Avg'
}

print(f"Indicators to analyze: {len(indicators)}")
print(f"Comparator groups: {len(comparator_groups)}")
print(f"Total comparisons: {len(indicators) * len(comparator_groups)}")

Indicators to analyze: 6
Comparator groups: 6
Total comparisons: 36


## 3. F-Test Implementation

Test for equality of variances between Iceland and each comparator group.
Formula: F = var₁/var₂ with two-tailed p-value

In [10]:
def calculate_f_test_with_significance(iceland_data, comparator_data, iceland_name="Iceland", comparator_name="Comparator"):
    """
    Calculate F-statistic with significance stars.
    Source: cs4_statistical_analysis.py lines 202-216
    """
    # Remove NaN values
    s1 = iceland_data.dropna()
    s2 = comparator_data.dropna()
    
    # Check sufficient data
    if len(s1) < 2 or len(s2) < 2:
        return {
            'f_statistic': np.nan,
            'p_value': np.nan,
            'significance': '',
            'variance_iceland': np.nan,
            'variance_comparator': np.nan
        }
    
    # Calculate variances (sample variance with ddof=1)
    var1 = np.var(s1, ddof=1)
    var2 = np.var(s2, ddof=1)
    
    # F-statistic (larger variance in numerator for two-tailed test)
    if var1 >= var2:
        f_stat = var1 / var2
        df1, df2 = len(s1) - 1, len(s2) - 1
    else:
        f_stat = var2 / var1
        df1, df2 = len(s2) - 1, len(s1) - 1
    
    # Two-tailed p-value
    p_value = 2 * min(stats.f.cdf(f_stat, df1, df2), 1 - stats.f.cdf(f_stat, df1, df2))
    
    # Determine significance
    if p_value < 0.01:
        significance = '***'
    elif p_value < 0.05:
        significance = '**'
    elif p_value < 0.10:
        significance = '*'
    else:
        significance = ''
    
    return {
        'f_statistic': f_stat,
        'p_value': p_value,
        'significance': significance,
        'variance_iceland': var1,
        'variance_comparator': var2,
        'iceland_higher': var1 > var2
    }

print("F-test function defined with significance levels:")
print("*** p < 0.01 (highly significant)")
print("**  p < 0.05 (significant)")
print("*   p < 0.10 (marginally significant)")

F-test function defined with significance levels:
*** p < 0.01 (highly significant)
**  p < 0.05 (significant)
*   p < 0.10 (marginally significant)


## 4. AR(4) Model Implementation

Fit autoregressive model of order 4: y_t = c + φ₁y_{t-1} + φ₂y_{t-2} + φ₃y_{t-3} + φ₄y_{t-4} + ε_t

In [11]:
def fit_ar4_transparent(series, series_name=""):
    """
    Fit AR(4) model with transparent output.
    Source: cs4_statistical_analysis.py lines 277-281
    """
    # Clean data
    clean_series = series.dropna()
    
    print(f"\n{series_name}:")
    print(f"  Data points: {len(clean_series)}")
    
    if len(clean_series) < 8:
        print(f"  ⚠️ Insufficient data for AR(4) model (need at least 8 points)")
        return None
    
    try:
        # Fit AR(4) model
        model = AutoReg(clean_series, lags=4, trend='c')
        fitted_model = model.fit()
        
        # Extract coefficients (excluding constant term)
        ar_coeffs = fitted_model.params[1:5].values  # φ₁, φ₂, φ₃, φ₄
        
        print(f"  AR(4) Coefficients:")
        for i, coeff in enumerate(ar_coeffs, 1):
            print(f"    φ{i} = {coeff:8.4f}")
        print(f"  AIC: {fitted_model.aic:8.2f}")
        print(f"  BIC: {fitted_model.bic:8.2f}")
        
        return {
            'coefficients': ar_coeffs,
            'aic': fitted_model.aic,
            'bic': fitted_model.bic,
            'n_obs': len(clean_series)
        }
        
    except Exception as e:
        print(f"  ❌ Error fitting AR(4): {str(e)}")
        return None

print("AR(4) model function defined")
print("Model: y_t = c + φ₁y_{t-1} + φ₂y_{t-2} + φ₃y_{t-3} + φ₄y_{t-4} + ε_t")

AR(4) model function defined
Model: y_t = c + φ₁y_{t-1} + φ₂y_{t-2} + φ₃y_{t-3} + φ₄y_{t-4} + ε_t


## 5. Impulse Response Half-Life Calculation

Calculate how many quarters it takes for a unit shock to decay to 50% of its initial value

In [12]:
def calculate_half_life(ar_coefficients):
    """
    Calculate half-life from AR(4) impulse response function.
    Source: cs4_statistical_analysis.py lines 296-340
    """
    if ar_coefficients is None or len(ar_coefficients) != 4:
        return "N/A"
    
    # Generate impulse response
    impulse_response = [1.0]  # Initial shock = 1
    
    # Track response over 20 quarters
    for t in range(1, 21):
        response = 0
        for lag in range(min(4, t)):
            if t-1-lag >= 0:
                response += ar_coefficients[lag] * impulse_response[t-1-lag]
        impulse_response.append(response)
    
    # Find when response ≤ 0.5 (50% of initial shock)
    for quarter, response in enumerate(impulse_response):
        if abs(response) <= 0.5:
            return quarter
    
    return "N/A"  # No half-life found within 20 quarters

print("Half-life calculation function defined")
print("Half-life = quarters until shock decays to ≤50% of initial value")

Half-life calculation function defined
Half-life = quarters until shock decays to ≤50% of initial value


## 6. RMSE Prediction Implementation

Rolling prediction methodology:
1. Train on all data except last 4 quarters
2. Predict last 4 quarters
3. Calculate RMSE = √(Σ(actual - predicted)² / 4)

In [13]:
def calculate_rmse_prediction(series, series_name="", forecast_periods=4):
    """
    Calculate RMSE using rolling prediction methodology.
    Source: cs4_statistical_analysis.py lines 350-384
    """
    clean_series = series.dropna()
    
    print(f"\n{series_name} RMSE Calculation:")
    print(f"  Total observations: {len(clean_series)}")
    
    if len(clean_series) < 8:
        print(f"  ⚠️ Insufficient data for RMSE calculation")
        return np.nan
    
    try:
        # Split data: training (all except last 4) and test (last 4)
        n_train = len(clean_series) - forecast_periods
        train_data = clean_series.iloc[:n_train]
        test_data = clean_series.iloc[n_train:]
        
        print(f"  Training set: {len(train_data)} observations")
        print(f"  Test set: {len(test_data)} observations")
        
        if len(train_data) < 8:
            print(f"  ⚠️ Insufficient training data")
            return np.nan
        
        # Fit AR(4) model on training data
        model = AutoReg(train_data, lags=4, trend='c')
        fitted_model = model.fit()
        
        # Generate forecast for test period
        forecast = fitted_model.forecast(steps=len(test_data))
        
        # Show actual vs predicted
        print(f"\n  Actual vs Predicted (last {len(test_data)} quarters):")
        for i, (actual, pred) in enumerate(zip(test_data.values, forecast.values)):
            error = actual - pred
            print(f"    Q{i+1}: Actual={actual:8.4f}, Predicted={pred:8.4f}, Error={error:8.4f}")
        
        # Calculate RMSE
        rmse = np.sqrt(np.mean((test_data.values - forecast.values)**2))
        print(f"\n  RMSE: {rmse:.6f}")
        
        return rmse
        
    except Exception as e:
        print(f"  ❌ Error calculating RMSE: {str(e)}")
        return np.nan

print("RMSE prediction function defined")
print("Methodology: Train on all except last 4 quarters, predict last 4, calculate RMSE")

RMSE prediction function defined
Methodology: Train on all except last 4 quarters, predict last 4, calculate RMSE


## 7. Load and Process Data for Each Indicator

In [14]:
# Store all results
all_results = []

for indicator_name, filename in indicators.items():
    print(f"\n{'='*70}")
    print(f"INDICATOR: {indicator_name}")
    print(f"{'='*70}")
    
    # Load data file
    file_path = data_dir / filename
    
    try:
        df = pd.read_csv(file_path)
        print(f"\nLoaded data: {df.shape[0]} observations")
        
        # Create date column for time series
        df['Date'] = pd.to_datetime(df['YEAR'].astype(str) + 'Q' + df['QUARTER'].astype(str))
        df = df.sort_values('Date')
        
        # Get Iceland data
        iceland_data = df['iceland_pgdp'].dropna()
        print(f"Iceland data points: {len(iceland_data)}")
        
    except FileNotFoundError:
        print(f"⚠️ File not found: {file_path}")
        print("Skipping this indicator...")
        continue
    except KeyError as e:
        print(f"⚠️ Missing column: {e}")
        print("Skipping this indicator...")
        continue
    
    # Process each comparator group
    for group in comparator_groups:
        if group not in df.columns:
            print(f"\n⚠️ Group '{group}' not found in data")
            continue
            
        print(f"\n{'-'*50}")
        print(f"Comparing Iceland vs {group_labels[group]}")
        print(f"{'-'*50}")
        
        comparator_data = df[group].dropna()
        print(f"Comparator data points: {len(comparator_data)}")
        
        # Initialize result record
        result = {
            'Indicator': indicator_name,
            'Comparator': group_labels[group],
            'Group_Code': group
        }
        
        # 1. F-TEST
        print(f"\n1. F-TEST FOR VARIANCE EQUALITY:")
        f_test = calculate_f_test_with_significance(iceland_data, comparator_data, 
                                                    "Iceland", group_labels[group])
        
        print(f"   Iceland variance: {f_test['variance_iceland']:.6f}")
        print(f"   {group_labels[group]} variance: {f_test['variance_comparator']:.6f}")
        print(f"   F-statistic: {f_test['f_statistic']:.4f}")
        print(f"   P-value: {f_test['p_value']:.6f}")
        print(f"   Significance: {f_test['significance']}")
        print(f"   Iceland higher volatility: {f_test['iceland_higher']}")
        
        result.update({
            'F_Statistic': f_test['f_statistic'],
            'P_Value': f_test['p_value'],
            'Significance': f_test['significance'],
            'Iceland_Higher': f_test['iceland_higher']
        })
        
        # 2. AR(4) MODEL & HALF-LIFE (Iceland only)
        if group == comparator_groups[0]:  # Only calculate once per indicator
            print(f"\n2. AR(4) MODEL (Iceland):")
            ar4_result = fit_ar4_transparent(iceland_data, "Iceland")
            
            if ar4_result:
                # Calculate half-life
                half_life = calculate_half_life(ar4_result['coefficients'])
                print(f"\n3. IMPULSE RESPONSE HALF-LIFE:")
                print(f"   Half-life: {half_life} quarters")
                
                result['AR4_Coefficients'] = ar4_result['coefficients'].tolist()
                result['AR4_AIC'] = ar4_result['aic']
                result['AR4_BIC'] = ar4_result['bic']
                result['Half_Life'] = half_life
            
            # 3. RMSE PREDICTION (Iceland)
            print(f"\n4. RMSE ROLLING PREDICTION (Iceland):")
            rmse = calculate_rmse_prediction(iceland_data, "Iceland")
            result['RMSE_Iceland'] = rmse
        
        # Store result
        all_results.append(result)

print(f"\n{'='*70}")
print(f"ANALYSIS COMPLETE")
print(f"Total comparisons: {len(all_results)}")


INDICATOR: Net Direct Investment

Loaded data: 105 observations
Iceland data points: 105

--------------------------------------------------
Comparing Iceland vs Eurozone Weighted Avg
--------------------------------------------------
Comparator data points: 105

1. F-TEST FOR VARIANCE EQUALITY:
   Iceland variance: 325.428499
   Eurozone Weighted Avg variance: 5.160456
   F-statistic: 63.0620
   P-value: 0.000000
   Significance: ***
   Iceland higher volatility: True

2. AR(4) MODEL (Iceland):

Iceland:
  Data points: 105
  AR(4) Coefficients:
    φ1 =   0.1694
    φ2 =   0.0568
    φ3 =   0.1938
    φ4 =  -0.2191
  AIC:   875.17
  BIC:   890.87

3. IMPULSE RESPONSE HALF-LIFE:
   Half-life: 1 quarters

4. RMSE ROLLING PREDICTION (Iceland):

Iceland RMSE Calculation:
  Total observations: 105
  Training set: 101 observations
  Test set: 4 observations

  Actual vs Predicted (last 4 quarters):
    Q1: Actual= -2.0122, Predicted= -3.6140, Error=  1.6018
    Q2: Actual= -1.6503, Predict

## 8. Create Summary Tables

In [15]:
# Convert results to DataFrame
results_df = pd.DataFrame(all_results)

# 1. F-Test Summary Table
print("\n" + "="*80)
print("F-TEST SUMMARY TABLE")
print("="*80)

# Pivot for F-statistics
f_stat_pivot = results_df.pivot_table(
    index='Indicator',
    columns='Comparator',
    values='F_Statistic',
    aggfunc='first'
)

# Add significance markers
significance_pivot = results_df.pivot_table(
    index='Indicator',
    columns='Comparator',
    values='Significance',
    aggfunc='first'
)

# Combine F-statistics with significance stars
f_test_summary = pd.DataFrame()
for col in f_stat_pivot.columns:
    f_test_summary[col] = f_stat_pivot[col].apply(lambda x: f"{x:.3f}" if pd.notna(x) else "N/A")
    # Add significance stars
    for idx in f_stat_pivot.index:
        if pd.notna(significance_pivot.loc[idx, col]) and significance_pivot.loc[idx, col]:
            f_test_summary.loc[idx, col] += significance_pivot.loc[idx, col]

print(f_test_summary.to_string())
print("\nNote: *** p<0.01, ** p<0.05, * p<0.10")


F-TEST SUMMARY TABLE
                                                           Baltics Simple Avg Baltics Weighted Avg Eurozone Simple Avg Eurozone Weighted Avg SOE Simple Avg SOE Weighted Avg
Indicator                                                                                                                                                                   
Net Capital Flows                                                   12.307***            12.557***           87.190***            175.475***       4.705***         3.126***
Net Direct Investment                                               30.874***            38.685***           25.382***             63.062***          1.094          1.662**
Net Other Investment                                                12.962***            13.397***           31.640***             76.255***       5.190***         2.829***
Net Portfolio Investment                                            38.078***            42.975***           30.8

In [16]:
# 2. AR(4) and Half-Life Summary
print("\n" + "="*80)
print("AR(4) MODEL AND HALF-LIFE SUMMARY (Iceland)")
print("="*80)

# Get unique AR(4) results per indicator
ar4_summary = results_df.groupby('Indicator').first()[['AR4_AIC', 'AR4_BIC', 'Half_Life', 'RMSE_Iceland']]
ar4_summary = ar4_summary.dropna(how='all')

# Format the display
ar4_display = pd.DataFrame()
ar4_display['AIC'] = ar4_summary['AR4_AIC'].apply(lambda x: f"{x:.2f}" if pd.notna(x) else "N/A")
ar4_display['BIC'] = ar4_summary['AR4_BIC'].apply(lambda x: f"{x:.2f}" if pd.notna(x) else "N/A")
ar4_display['Half-Life (quarters)'] = ar4_summary['Half_Life']
ar4_display['RMSE'] = ar4_summary['RMSE_Iceland'].apply(lambda x: f"{x:.6f}" if pd.notna(x) else "N/A")

print(ar4_display.to_string())


AR(4) MODEL AND HALF-LIFE SUMMARY (Iceland)
                                                               AIC      BIC  Half-Life (quarters)       RMSE
Indicator                                                                                                   
Net Capital Flows                                           954.63   970.32                   1.0   7.606227
Net Direct Investment                                       875.17   890.87                   1.0  15.334450
Net Other Investment                                        991.50  1007.19                   1.0   4.954051
Net Portfolio Investment                                    956.19   971.88                   1.0  15.129463
Net Portfolio Investment - Debt Securities                  934.91   950.60                   1.0  13.226917
Net Portfolio Investment - Equity & Investment Fund Shares  772.42   788.11                   1.0   7.337777


In [17]:
# 3. Count of significant results
print("\n" + "="*80)
print("STATISTICAL SIGNIFICANCE SUMMARY")
print("="*80)

# Count significant results by comparator
for comparator in results_df['Comparator'].unique():
    comp_data = results_df[results_df['Comparator'] == comparator]
    
    total = len(comp_data)
    sig_01 = len(comp_data[comp_data['P_Value'] < 0.01])
    sig_05 = len(comp_data[(comp_data['P_Value'] >= 0.01) & (comp_data['P_Value'] < 0.05)])
    sig_10 = len(comp_data[(comp_data['P_Value'] >= 0.05) & (comp_data['P_Value'] < 0.10)])
    not_sig = len(comp_data[comp_data['P_Value'] >= 0.10])
    
    print(f"\n{comparator}:")
    print(f"  Highly significant (p<0.01): {sig_01}/{total} indicators")
    print(f"  Significant (p<0.05): {sig_05}/{total} indicators")
    print(f"  Marginally significant (p<0.10): {sig_10}/{total} indicators")
    print(f"  Not significant (p≥0.10): {not_sig}/{total} indicators")
    
    # Iceland higher volatility
    iceland_higher = len(comp_data[comp_data['Iceland_Higher'] == True])
    print(f"  Iceland higher volatility: {iceland_higher}/{total} indicators")


STATISTICAL SIGNIFICANCE SUMMARY

Eurozone Weighted Avg:
  Highly significant (p<0.01): 6/6 indicators
  Significant (p<0.05): 0/6 indicators
  Marginally significant (p<0.10): 0/6 indicators
  Not significant (p≥0.10): 0/6 indicators
  Iceland higher volatility: 6/6 indicators

Eurozone Simple Avg:
  Highly significant (p<0.01): 6/6 indicators
  Significant (p<0.05): 0/6 indicators
  Marginally significant (p<0.10): 0/6 indicators
  Not significant (p≥0.10): 0/6 indicators
  Iceland higher volatility: 6/6 indicators

SOE Weighted Avg:
  Highly significant (p<0.01): 5/6 indicators
  Significant (p<0.05): 1/6 indicators
  Marginally significant (p<0.10): 0/6 indicators
  Not significant (p≥0.10): 0/6 indicators
  Iceland higher volatility: 4/6 indicators

SOE Simple Avg:
  Highly significant (p<0.01): 5/6 indicators
  Significant (p<0.05): 0/6 indicators
  Marginally significant (p<0.10): 0/6 indicators
  Not significant (p≥0.10): 1/6 indicators
  Iceland higher volatility: 4/6 indicat

## 9. Save Results to CSV

In [18]:
# Save detailed results
output_dir = Path('../outputs')
output_dir.mkdir(exist_ok=True)

# Full results
results_df.to_csv(output_dir / 'CS4_detailed_results.csv', index=False)
print(f"Detailed results saved to: {output_dir / 'CS4_detailed_results.csv'}")

# F-test summary
f_test_summary.to_csv(output_dir / 'CS4_f_test_summary.csv')
print(f"F-test summary saved to: {output_dir / 'CS4_f_test_summary.csv'}")

# AR(4) summary
ar4_display.to_csv(output_dir / 'CS4_ar4_summary.csv')
print(f"AR(4) summary saved to: {output_dir / 'CS4_ar4_summary.csv'}")

print("\n✅ CS4 Statistical Analysis Complete!")

Detailed results saved to: ../outputs/CS4_detailed_results.csv
F-test summary saved to: ../outputs/CS4_f_test_summary.csv
AR(4) summary saved to: ../outputs/CS4_ar4_summary.csv

✅ CS4 Statistical Analysis Complete!


## 10. Methodological Notes

### F-Test Methodology
- **Null Hypothesis**: σ²₁ = σ²₂ (equal variances)
- **Test Statistic**: F = max(var₁, var₂) / min(var₁, var₂)
- **P-value**: Two-tailed test using F-distribution
- **Interpretation**: Reject H₀ if p < 0.05 → variances are significantly different

### AR(4) Model
- **Model**: y_t = c + φ₁y_{t-1} + φ₂y_{t-2} + φ₃y_{t-3} + φ₄y_{t-4} + ε_t
- **Purpose**: Capture persistence in capital flow volatility
- **Information Criteria**: Lower AIC/BIC indicates better model fit

### Half-Life Calculation
- **Definition**: Quarters until unit shock decays to ≤50%
- **Method**: Generate impulse response function from AR(4) coefficients
- **Interpretation**: Shorter half-life = faster mean reversion

### RMSE Methodology
- **Training**: All data except last 4 quarters
- **Testing**: Last 4 quarters
- **Metric**: RMSE = √(mean squared prediction error)
- **Interpretation**: Lower RMSE = better predictive accuracy